In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

In [2]:
import numpy as np


SCHEME_OFFSETS = {
    "BPSK":  0,
    "QPSK":  2,       # 0 + 2
    "16QAM": 6,       # 2 + 4
    "64QAM": 22,      # 6 + 16
}
TOTAL_CLASSES = 86    # 2 + 4 + 16 + 64

# (i_offset, i_step, q_offset, q_step) for converting amplitude values → table indices
SCHEME_NORMALIZATION = {
    "BPSK":  (1, 2, 0, 1),
    "QPSK":  (1, 2, 1, 2),
    "16QAM": (3, 2, 3, 2),
    "64QAM": (7, 2, 7, 2),
}


def build_lookup_table(symbol_map, i_levels, q_levels, global_offset):
    """Build a 2D numpy lookup table mapping (I, Q) constellation points to global class indices.

    Each scheme occupies a contiguous slice of the shared softmax vector:
        BPSK  → [0,  2)
        QPSK  → [2,  6)
        16QAM → [6,  22)
        64QAM → [22, 86)

    The stored value is bits + global_offset, so the index is valid directly
    against the full 86-class softmax output.

    Constellation values are mapped to array indices via:
        idx = (val + offset) // step
    where offset = -levels[0] and step = levels[1] - levels[0] (defaulting to
    1 when only a single level exists, as in the BPSK Q axis).

    Args:
        symbol_map: Dict mapping (i_val, q_val) tuples to integer bit patterns.
        i_levels: Tuple of valid I-axis amplitude values in ascending order.
        q_levels: Tuple of valid Q-axis amplitude values in ascending order.
        global_offset: Integer offset (from SCHEME_OFFSETS) to shift local bit
            pattern values into the global class index space.

    Returns:
        np.ndarray of shape (len(i_levels), len(q_levels)) with dtype uint8,
        where table[i_idx, q_idx] is the global class index for that point.
    """
    i_offset = -i_levels[0]
    i_step = (i_levels[1] - i_levels[0]) if len(i_levels) > 1 else 1
    q_offset = -q_levels[0]
    q_step = (q_levels[1] - q_levels[0]) if len(q_levels) > 1 else 1

    table = np.zeros((len(i_levels), len(q_levels)), dtype=np.uint8)
    for (i, q), bits in symbol_map.items():
        table[(i + i_offset) // i_step, (q + q_offset) // q_step] = bits + global_offset
    return table


def _build_global_index_to_symbol():
    """Build the global array mapping each class index to its binary symbol string.

    Indices 0–1 are BPSK symbols ("0", "1"), 2–5 are QPSK ("00"–"11"),
    6–21 are 16QAM ("0000"–"1111"), and 22–85 are 64QAM ("000000"–"111111").
    All strings are stored with dtype '<U6' (width of the longest symbol).

    Returns:
        np.ndarray of shape (86,) with dtype '<U6'.
    """
    entries = (
        [(i, 1) for i in range(2)]    # BPSK
        + [(i, 2) for i in range(4)]  # QPSK
        + [(i, 4) for i in range(16)] # 16QAM
        + [(i, 6) for i in range(64)] # 64QAM
    )
    return np.array([format(i, f'0{n}b') for i, n in entries], dtype='<U6')


index_to_symbol = _build_global_index_to_symbol()


scheme_to_high_low_map = {
    "BPSK": [(-1, 1), (0,1)],
    "QPSK": [(-1, 1), (-1, 1)],
    "16QAM": [(-2, 2), (-2, 2)],
    "64QAM": [(-4, 4), (-4, 4)],
}

_bpsk_i_levels = (-1, 1)
_bpsk_q_levels = (0,)
bpsk_map = {
    (-1, 0): 0b0,
    (1, 0): 0b1,
}
bpsk_table = build_lookup_table(bpsk_map, _bpsk_i_levels, _bpsk_q_levels, SCHEME_OFFSETS["BPSK"])

_qpsk_levels = (-1, 1)
qpsk_map = {
    (-1, -1): 0b00,
    (-1, 1): 0b01,
    (1, 1): 0b11,
    (1, -1): 0b10,
}
qpsk_table = build_lookup_table(qpsk_map, _qpsk_levels, _qpsk_levels, SCHEME_OFFSETS["QPSK"])

_qam16_levels = (-3, -1, 1, 3)
_qam16_gray = (0b00, 0b01, 0b11, 0b10)
qam16_map = {
    (i, q): (i_bits << 2) | q_bits
    for i, i_bits in zip(_qam16_levels, _qam16_gray)
    for q, q_bits in zip(_qam16_levels, _qam16_gray)
}
qam16_table = build_lookup_table(qam16_map, _qam16_levels, _qam16_levels, SCHEME_OFFSETS["16QAM"])

_qam64_levels = (-7, -5, -3, -1, 1, 3, 5, 7)
_qam64_gray = (0b000, 0b001, 0b011, 0b010, 0b110, 0b111, 0b101, 0b100)
qam64_map = {
    (i, q): (i_bits << 3) | q_bits
    for i, i_bits in zip(_qam64_levels, _qam64_gray)
    for q, q_bits in zip(_qam64_levels, _qam64_gray)
}
qam64_table = build_lookup_table(qam64_map, _qam64_levels, _qam64_levels, SCHEME_OFFSETS["64QAM"])

scheme_to_index_table_map = {
    "BPSK": bpsk_table,
    "QPSK": qpsk_table,
    "16QAM": qam16_table,
    "64QAM": qam64_table,
}



In [3]:
from typing import Union

import torch
from numpy import ndarray
from torch.utils.data import Dataset, Subset


class IQDataset(Dataset):
    def __init__(self, data: Union[ndarray, torch.Tensor], labels: Union[ndarray, torch.Tensor], transform=None):
        """
        Args:
            data: np.ndarray of shape (n_samples, length, 2), where axis 2 holds the I sample at index 0 and the Q sample at index 1.
            labels: np.ndarray of shape (n_samples,) containing the corresponding modulation scheme labels for each IQ sequence.
        """
        self.data: torch.Tensor = torch.from_numpy(data) if isinstance(data, ndarray) else data
        self.labels: torch.Tensor = torch.from_numpy(labels) if isinstance(labels, ndarray) else labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            sample = self.transform(sample)

        return sample, label

In [4]:
from scipy.signal import butter, sosfilt


def add_noise(signal: IQDataset,
              snr_db: float = None,
              rng: np.random.Generator = None) -> IQDataset:
    """Add AWGN (Additive White Gaussian Noise) to an IQ dataset.

    Noise power is derived per signal from its measured power and the target
    SNR, then split equally across I and Q channels.

    Args:
        signal: IQDataset whose data tensor has shape (B, L, 2).
        snr_db: Signal-to-noise ratio in decibels. If None, sampled uniformly
            from [5, 15] dB.
        rng: NumPy random generator. If None, a fresh default generator is used.

    Returns:
        New IQDataset with float32 noisy signals and the original labels.
    """
    if rng is None:
        rng = np.random.default_rng()
    if snr_db is None:
        snr_db = np.random.uniform(5, 15)

    # find signal power
    signal_data = signal.data.numpy()
    signal_power = np.mean(np.sum(signal_data**2, axis=2, keepdims=True), axis=1, keepdims=True)

    # find noise power and std deviation
    noise_powers = signal_power / (10 ** (snr_db / 10))
    noise_std = np.sqrt(noise_powers / 2)
    noise = rng.normal(0, noise_std, signal_data.shape)

    return IQDataset(signal_data + noise, signal.labels)


def add_interference(signal: IQDataset,
                     interference_signal: IQDataset,
                     interference_ratio: float = None) -> IQDataset:
    """Add a scaled interfering signal to an IQ dataset.

    Models a second transmitter whose signal is received simultaneously.
    The interference is added directly to the IQ samples; labels are unchanged
    since they reflect the intended transmitted symbols.

    Args:
        signal: IQDataset to corrupt, with data shape (B, L, 2).
        interference_signal: IQDataset used as the interferer. Must have the
            same shape as signal.
        interference_ratio: Scaling factor applied to the interferer before
            addition. If None, sampled uniformly from [0, 0.3].

    Returns:
        New IQDataset with the combined signal and the original labels.
    """
    if interference_ratio is None:
        interference_ratio = np.random.uniform(0, 0.3)

    scaled_interference = interference_signal.data * interference_ratio
    combined_signal = signal.data + scaled_interference
    return IQDataset(combined_signal, signal.labels)


def apply_low_pass_filter(signal: IQDataset) -> IQDataset:
    """Apply a 4th-order Butterworth low-pass filter to an IQ dataset.

    Models the band-limiting effect of real hardware (antennas, amplifiers,
    cables). The filter is applied along the time/symbol axis (axis=1),
    causing adjacent symbols to bleed into each other (inter-symbol
    interference). Uses second-order sections (SOS) for numerical stability.

    Args:
        signal: IQDataset whose data tensor has shape (B, L, 2).

    Returns:
        New IQDataset with float32 filtered signals and the original labels.
    """
    # TODO: COME BACK AND RESEARCH

    # Convert to numpy
    np_data = signal.data.numpy()

    # gen filter
    sos = butter(N=4, Wn=0.4, btype='low', analog=False, output='sos')
    filtered = sosfilt(sos, np_data, axis=1)

    return IQDataset(torch.from_numpy(filtered).to(dtype=torch.float32), signal.labels)


In [5]:
from typing import Literal, List, Tuple



class IQGenerator:
    """Uses a seeded NumPy random generator so that datasets are reproducible.
    Supported schemes: BPSK, QPSK, 16QAM, 64QAM.
    """

    def __init__(self, seed: int = 42, scheme_distribution = (0.25,0.25,0.25,0.25)):
        """
        Args:
            seed: Seed for the default NumPy random generator. Defaults to 42.
            scheme_distribution: Percentage distribution of modulation schemes in generated datasets. Should be a list of 4 integers summing to 100, corresponding to BPSK, QPSK, 16QAM, and 64QAM respectively. Defaults to an even distribution.
        """
        self.rng = np.random.default_rng(seed)
        self.scheme_distribution = scheme_distribution
        if sum(scheme_distribution) != 1 or len(scheme_distribution) != 4:
            raise ValueError("scheme_distribution must be a list of 4 floats summing to 1.")

    def _get_scheme_boundaries(self):
        """Compute cumulative probability boundaries for each modulation scheme.

        Returns:
            Tuple of (first_bound, second_bound, third_bound) where each value
            is the upper boundary of the corresponding scheme's probability range:
            BPSK → [0, first), QPSK → [first, second), 16QAM → [second, third),
            64QAM → [third, 1].
        """
        bpsk_dist, qpsk_dist, sixteen_qam_dist, sixtyfour_qam_dist = self.scheme_distribution
        first_bound = bpsk_dist
        second_bound = first_bound + qpsk_dist
        third_bound = second_bound + sixteen_qam_dist
        return first_bound, second_bound, third_bound

    def _get_scheme_masks(self, num_samples=128):
        """Generate boolean masks assigning each sample to a modulation scheme.

        Draws a uniform random value per sample and partitions samples into
        schemes according to scheme_distribution.

        Args:
            num_samples: Number of samples to assign. Defaults to 128.

        Returns:
            Tuple of four boolean arrays (bpsk_mask, qpsk_mask, stqam_mask,
            sfqam_mask), each of shape (num_samples,).
        """
        first_bound, second_bound, third_bound = self._get_scheme_boundaries()
        uni = self.rng.random(size=(num_samples,))
        bpsk_mask = uni < first_bound
        qpsk_mask = (uni >= first_bound) & (uni < second_bound)
        stqam_mask = (uni >= second_bound) & (uni < third_bound)
        sfqam_mask = uni >= third_bound
        return bpsk_mask, qpsk_mask, stqam_mask, sfqam_mask

    def _allocate_iq_and_label_arrays(self, num_samples, length):
        """Allocate zeroed arrays for IQ signals and symbol label indices.

        Args:
            num_samples: Number of signal sequences.
            length: Number of symbols per sequence.

        Returns:
            Tuple of (iq_arr, symbol_indices_arr) where iq_arr has shape
            (num_samples, length, 2) with dtype float32, and symbol_indices_arr
            has shape (num_samples, length) with dtype int64.
        """
        iq_arr = np.zeros(shape=(num_samples, length, 2), dtype=np.float32)
        symbol_indices_arr = np.zeros(shape=(num_samples, length), dtype=np.int64)
        return iq_arr, symbol_indices_arr

    def _generate_mask_scheme_pairs(self, num_samples):
        """Pair each scheme's boolean mask with its scheme name.

        Args:
            num_samples: Number of samples to assign across schemes.

        Returns:
            List of (mask, scheme) tuples where mask is a boolean array of
            shape (num_samples,) and scheme is the corresponding scheme string.
        """
        bpsk_mask, qpsk_mask, stqam_mask, sfqam_mask = self._get_scheme_masks(num_samples=num_samples)
        pairs: List [Tuple[ndarray, Literal["BPSK", "QPSK", "16QAM", "64QAM"]]] = [
            (bpsk_mask, "BPSK"),
            (qpsk_mask, "QPSK"),
            (stqam_mask, "16QAM"),
            (sfqam_mask, "64QAM"),
        ]
        return pairs

    def generate_signals(self, n_samples=128, length=256, seed=None, modulation_scheme: Literal["BPSK", "QPSK", "16QAM", "64QAM"] = "BPSK"):
        """Randomly draws constellation point indices, maps them to odd-integer
        amplitude levels (e.g. ±1 for BPSK, ±1/±3 for QPSK), and stacks I
        and Q channels.

        Args:
            n_samples: Number of independent IQ sequences to generate.
            length: Number of symbols per sequence. Defaults to 256.
            seed: Optional per-call seed. When provided, a fresh generator is
                created for this call only, leaving the instance generator
                unchanged. Defaults to None (use the instance generator).
            modulation_scheme: One of "BPSK", "QPSK", "16QAM", or "64QAM".

        Returns:
            np.ndarray of shape (n_samples, length, 2), where axis 2 holds
            the I sample at index 0 and the Q sample at index 1.
        """
        # in case you want to use the same generator to create different datasets
        rand = self.rng
        if seed is not None:
            rand = np.random.default_rng(seed)

        # get boundaries for samples
        i_bounds, q_bounds = scheme_to_high_low_map[modulation_scheme]
        i_low, i_high = i_bounds
        q_low, q_high = q_bounds

        i_samples = rand.integers(i_low, i_high, size=(n_samples, length))
        q_samples = rand.integers(q_low, q_high, size=(n_samples, length))

        # convert into proper format for IQ
        i_samples = 2 * i_samples + 1
        if modulation_scheme != "BPSK":
            q_samples = 2 * q_samples + 1

        # stack to create distinct channels
        return np.stack((i_samples, q_samples), axis=2).astype(np.float32)

    def generate_softmax_indices_for_signals(self, iq_signals, modulation_scheme: Literal["BPSK", "QPSK", "16QAM", "64QAM"] = "BPSK"):
        """Map IQ signal amplitudes to global softmax class indices.

        Converts raw I and Q amplitude values to table indices using per-scheme
        normalization, then looks up the global class index for each (I, Q) pair.

        Args:
            iq_signals: np.ndarray of shape (n_samples, length, 2) as returned
                by generate_signals.
            modulation_scheme: One of "BPSK", "QPSK", "16QAM", or "64QAM".

        Returns:
            np.ndarray of shape (n_samples, length) with dtype int64 containing
            global class indices in the range defined by SCHEME_OFFSETS.
        """
        index_table = scheme_to_index_table_map[modulation_scheme]
        i_offset, i_step, q_offset, q_step = SCHEME_NORMALIZATION[modulation_scheme]
        i_idx = (iq_signals[:, :, 0].astype(int) + i_offset) // i_step
        q_idx = (iq_signals[:, :, 1].astype(int) + q_offset) // q_step
        return index_table[i_idx, q_idx].astype(np.int64)

    def generate_dataset(self, num_samples=128, length=128, seed=None) -> IQDataset:
        """Generate a mixed-scheme IQDataset ready for training.

        Samples are assigned to modulation schemes according to
        scheme_distribution, then IQ signals and their corresponding global
        softmax label indices are generated for each scheme.

        Args:
            num_samples: Total number of signal sequences in the dataset.
                Defaults to 128.
            length: Number of symbols per sequence. Defaults to 128.
            seed: Optional per-call seed. When provided, a fresh generator is
                created for this call only, leaving the instance generator
                unchanged. Defaults to None (use the instance generator).

        Returns:
            IQDataset with data of shape (num_samples, length, 2) and labels
            of shape (num_samples, length) containing global class indices.
        """
        # allocate memory for the dataset
        iq_arr, symbol_indices_arr = self._allocate_iq_and_label_arrays(num_samples=num_samples, length=length)

        # map masks to schemes for generation purposes
        mask_scheme_pairs = self._generate_mask_scheme_pairs(num_samples=num_samples)

        # loop over each mask and set to a generate signal. Set labels based on the scheme
        for mask, scheme in mask_scheme_pairs:
            # number of samples to generate for this scheme is the number of True values in the mask
            count = mask.sum()
            if count == 0:
                continue
            # set those rows to IQ signals
            iq_arr[mask] = self.generate_signals(n_samples=count, length=length, modulation_scheme=scheme, seed=seed)
            symbol_indices_arr[mask] = self.generate_softmax_indices_for_signals(iq_arr[mask], modulation_scheme=scheme)

        return IQDataset(data=iq_arr, labels=symbol_indices_arr)

In [6]:
class AttentionModel(nn.Module):
    def __init__(self):
        # takes in (B,2,L) and outputs (B,1,L), or simply (B,L) after squeezing
        # each (I,Q) needs to attend to all other (I,Q) pairs in their signal. the sequence is the L dimension, with the vector being [I,Q]
        super().__init__()
        # (B,2,L) -> (B,64,L) for attention
        self.embed1 = nn.Linear(2, 64)  # embed (I,Q) pairs into a higher-dimensional space
        # (B,L,64) -> (B,L,64) for attention
        self.attention1 = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True, dropout=0.1) # attention computes (S,S) where each E_i,j is how much query i attends to key j. Then matmul by (S,dv), or all value vectors to take the weighted sum of values for each query. Concatenate all heads to be (S, dv*num_heads) = (S, embed_dim) since dv = embed_dim/num_heads
        self.linear1 = nn.Linear(64, 128)
        self.linear2 = nn.Linear(128, 64)
        self.attention2 = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True, dropout=0.1)
        self.linear3 = nn.Linear(64, 128)
        self.linear4 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 86)# one output per possible symbol

        self.layernorm1 = nn.LayerNorm(64)
        self.layernorm2 = nn.LayerNorm(64)
        self.layernorm3 = nn.LayerNorm(64)
        self.layernorm4 = nn.LayerNorm(64)

        self.gelu = nn.GELU()

    def forward(self, x):
        # x is (B,2,L)
        x = self.embed1(x) # (B,L,64) for attention)

        x = self.layernorm1(x + self.attention1(x,x,x)[0])
        y = self.linear1(x)
        y = self.gelu(y)
        y = self.linear2(y)
        x = self.layernorm2(x + y)
        x = self.layernorm3(x + self.attention2(x,x,x)[0])
        y = self.linear3(x)
        y = self.gelu(y)
        y = self.linear4(y)
        x = self.layernorm4(x + y)
        return self.output(x)




In [7]:
from torch.utils import data

initial_generator = IQGenerator(seed=42, scheme_distribution=(0.25, 0.25, 0.25, 0.25))
initial_dataset = initial_generator.generate_dataset(num_samples=2000, length=128)
initial_train = Subset(initial_dataset, indices=range(1750))
initial_validate = Subset(initial_dataset, indices=range(1750, 2000))

base_train_dataloader = data.DataLoader(initial_train, batch_size=64, shuffle=True)
base_val_dataloader = data.DataLoader(initial_validate, batch_size=64, shuffle=False)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'


In [8]:
def train(train_dataloader, val_dataloader, model, epochs=10, device=None):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x_batch, y_batch in train_dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch) # (B,2,L) for the model
            loss = loss_fn(outputs.permute(0,2,1), y_batch) # flatten to (B*L, 86) and (B*L,)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

        # Validation step can be added here if needed

In [9]:
train(model=AttentionModel(), train_dataloader=base_train_dataloader, val_dataloader=base_val_dataloader, epochs=10, device=device)

Epoch 1/10, Loss: 2.8805
Epoch 2/10, Loss: 1.4942
Epoch 3/10, Loss: 0.8295
Epoch 4/10, Loss: 0.4752
Epoch 5/10, Loss: 0.2796
Epoch 6/10, Loss: 0.1670
Epoch 7/10, Loss: 0.1051
Epoch 8/10, Loss: 0.0691
Epoch 9/10, Loss: 0.0477
Epoch 10/10, Loss: 0.0354
